# Top-5 ROC and PR Curves by Validation Strategy

This notebook uses the existing held-out prediction and model-performance files. It does not retrain models.

Figures generated:

1. AUROC curves for the top 5 model/feature-view combinations under repeated duplicate-aware holdout validation.
2. AUPRC curves for the same top 5 duplicate-aware combinations.
3. AUROC curves for the top 5 model/feature-view combinations under scaffold 5-fold cross-validation.
4. AUPRC curves for the same top 5 scaffold-CV combinations.

Duplicate-aware curves pool held-out predictions from repeated duplicate-aware splits. These are repeated holdout splits, not true cross-validation folds. Scaffold-CV curves pool out-of-fold predictions from the five scaffold folds.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve


def find_validation_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "brainroute_ml_validation", cwd.parent]
    for candidate in candidates:
        if (candidate / "reports" / "model_predictions_all_splits.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find brainroute_ml_validation reports directory from current working directory.")


VALIDATION_ROOT = find_validation_root()
REPORT_DIR = VALIDATION_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PERFORMANCE_PATH = REPORT_DIR / "model_performance_all_splits.csv"
PREDICTIONS_PATH = REPORT_DIR / "model_predictions_all_splits.csv"

performance = pd.read_csv(PERFORMANCE_PATH)
predictions = pd.read_csv(PREDICTIONS_PATH)

print(f"Loaded {len(performance):,} performance rows from {PERFORMANCE_PATH}")
print(f"Loaded {len(predictions):,} prediction rows from {PREDICTIONS_PATH}")

## Shared Plot Settings

In [ ]:
FIGSIZE = (8, 7)
DPI = 300
LINEWIDTH = 2.25
FONT_SIZE = 11
TITLE_SIZE = 13
LABEL_SIZE = 12

BLUE_PALETTE = [
    "#0B3D91",
    "#1769AA",
    "#2F80C1",
    "#5DADE2",
    "#8EC5F4",
]

plt.rcParams.update(
    {
        "figure.dpi": DPI,
        "savefig.dpi": DPI,
        "font.size": FONT_SIZE,
        "axes.titlesize": TITLE_SIZE,
        "axes.labelsize": LABEL_SIZE,
        "legend.fontsize": 8.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.7,
    }
)


def pretty_name(feature_view: str, model: str) -> str:
    view_labels = {
        "padel": "PaDEL",
        "morgan": "Morgan",
        "padel_morgan": "PaDEL + Morgan",
        "embeddings": "ChemBERTa",
        "padel_morgan_embeddings": "PaDEL + Morgan + ChemBERTa",
    }
    model_labels = {
        "logistic_regression": "Logistic Regression",
        "knn": "KNN",
        "random_forest": "Random Forest",
        "extra_trees": "Extra Trees",
        "lightgbm": "LightGBM",
        "xgboost": "XGBoost",
    }
    return f"{view_labels.get(feature_view, feature_view)} / {model_labels.get(model, model)}"


def top_combinations(split_prefix: str, n: int = 5) -> pd.DataFrame:
    subset = performance[performance["split"].str.startswith(split_prefix)].copy()
    ranked = (
        subset.groupby(["feature_view", "model"], as_index=False)
        .agg(
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            balanced_accuracy_std=("balanced_accuracy", "std"),
            auroc_mean=("roc_auc", "mean"),
            auprc_mean=("auprc", "mean"),
            n_splits=("split", "nunique"),
        )
        .sort_values(["balanced_accuracy_mean", "auprc_mean", "auroc_mean"], ascending=False)
        .head(n)
        .reset_index(drop=True)
    )
    return ranked


duplicate_top5 = top_combinations("duplicate_aware_seed")
scaffold_top5 = top_combinations("scaffold_cv_fold")

display(duplicate_top5)
display(scaffold_top5)

## Plot Functions

In [ ]:
def pooled_predictions(split_prefix: str, feature_view: str, model: str) -> pd.DataFrame:
    subset = predictions[
        predictions["split"].str.startswith(split_prefix)
        & (predictions["feature_view"] == feature_view)
        & (predictions["model"] == model)
    ].copy()
    subset = subset.dropna(subset=["y_true", "y_score"])
    return subset


def plot_roc(top5: pd.DataFrame, split_prefix: str, title: str, output_name: str) -> Path:
    fig, ax = plt.subplots(figsize=FIGSIZE)
    for color, row in zip(BLUE_PALETTE, top5.itertuples(index=False)):
        df = pooled_predictions(split_prefix, row.feature_view, row.model)
        if df["y_true"].nunique() < 2:
            continue
        fpr, tpr, _ = roc_curve(df["y_true"].astype(int), df["y_score"].astype(float))
        auc_value = roc_auc_score(df["y_true"].astype(int), df["y_score"].astype(float))
        label = f"{pretty_name(row.feature_view, row.model)} (AUROC={auc_value:.3f})"
        ax.plot(fpr, tpr, color=color, linewidth=LINEWIDTH, label=label)

    ax.plot([0, 1], [0, 1], color="#6B7280", linestyle="--", linewidth=1.25, label="Chance")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(title)
    ax.legend(loc="lower right", frameon=True, framealpha=0.95)
    fig.tight_layout()
    output_path = FIGURE_DIR / output_name
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    return output_path


def plot_pr(top5: pd.DataFrame, split_prefix: str, title: str, output_name: str) -> Path:
    fig, ax = plt.subplots(figsize=FIGSIZE)
    prevalence_values = []
    for color, row in zip(BLUE_PALETTE, top5.itertuples(index=False)):
        df = pooled_predictions(split_prefix, row.feature_view, row.model)
        if df["y_true"].nunique() < 2:
            continue
        y_true = df["y_true"].astype(int)
        y_score = df["y_score"].astype(float)
        precision, recall, _ = precision_recall_curve(y_true, y_score)
        ap_value = average_precision_score(y_true, y_score)
        prevalence_values.append(float(y_true.mean()))
        label = f"{pretty_name(row.feature_view, row.model)} (AUPRC={ap_value:.3f})"
        ax.plot(recall, precision, color=color, linewidth=LINEWIDTH, label=label)

    y_min = 0.0
    if prevalence_values:
        baseline = float(np.mean(prevalence_values))
        y_min = max(0.0, baseline - 0.08)
        ax.axhline(baseline, color="#6B7280", linestyle="--", linewidth=1.25, label=f"Baseline={baseline:.3f}")
    ax.set_xlim(0, 1)
    ax.set_ylim(y_min, 1.02)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title)
    ax.legend(loc="lower left", frameon=True, framealpha=0.95)
    fig.tight_layout()
    output_path = FIGURE_DIR / output_name
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    return output_path


## Generate Duplicate-Aware Figures

In [ ]:
duplicate_roc_path = plot_roc(
    duplicate_top5,
    "duplicate_aware_seed",
    "Top 5 AUROC Curves: Repeated Duplicate-Aware Holdout Validation",
    "top5_duplicate_aware_auroc_curves.png",
)

duplicate_pr_path = plot_pr(
    duplicate_top5,
    "duplicate_aware_seed",
    "Top 5 AUPRC Curves: Repeated Duplicate-Aware Holdout Validation",
    "top5_duplicate_aware_auprc_curves.png",
)

print(duplicate_roc_path)
print(duplicate_pr_path)

## Generate Scaffold-CV Figures

In [ ]:
scaffold_roc_path = plot_roc(
    scaffold_top5,
    "scaffold_cv_fold",
    "Top 5 AUROC Curves: Scaffold 5-Fold Cross-Validation",
    "top5_scaffold_cv_auroc_curves.png",
)

scaffold_pr_path = plot_pr(
    scaffold_top5,
    "scaffold_cv_fold",
    "Top 5 AUPRC Curves: Scaffold 5-Fold Cross-Validation",
    "top5_scaffold_cv_auprc_curves.png",
)

print(scaffold_roc_path)
print(scaffold_pr_path)

## Figure Check

In [ ]:
expected_files = [
    "top5_duplicate_aware_auroc_curves.png",
    "top5_duplicate_aware_auprc_curves.png",
    "top5_scaffold_cv_auroc_curves.png",
    "top5_scaffold_cv_auprc_curves.png",
]

for filename in expected_files:
    path = FIGURE_DIR / filename
    print(f"{filename}: {'created' if path.exists() else 'missing'}")

assert len(duplicate_top5) == 5
assert len(scaffold_top5) == 5
assert all((FIGURE_DIR / filename).exists() for filename in expected_files)